<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-langchain-00/fictional-department-rag-langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG (Retrieval-Augmented Generation) System
## Fictional Undergrad Department - DQE (Department of Quantum Engineering)

This notebook demonstrates building a RAG pipeline using **LangChain** for question-answering about a fictional department (DQE).

## What We'll Build:

1. **Document Embedding** - Convert text documents into vector representations using `all-MiniLM-L6-v2` via `langchain-huggingface`
2. **Vector Store** - Create a FAISS vector store using `langchain-community`
3. **Retrieval** - Find relevant documents using LangChain's retriever interface
4. **Generation** - Use a model-agnostic LLM (default: `gemini-2.5-flash`) via `init_chat_model`

## Key Technologies:
- **LangChain**: Orchestration framework for building RAG pipelines
- **HuggingFaceEmbeddings**: Open-source embedding model (runs locally)
- **FAISS**: Facebook's similarity search library
- **`init_chat_model`**: LangChain's model-agnostic initializer (supports Gemini, OpenAI, HuggingFace, and more)

## Setup Instructions

### Environment Variables

Add to Google Colab Secrets):

#### 1. Gemini API Key (`GEMINI_API_KEY`)
Used for the default LLM (`gemini-2.5-flash`).

1. Visit [https://aistudio.google.com/](https://aistudio.google.com/) and sign in
2. Click **Get API key → Create API key**
3. Add to google secrets
   ```
   GEMINI_API_KEY=<your secret>
   GOOGLE_API_KEY=<same secret here too>
   ```


---
### Step 1: Install Dependencies

Install all required LangChain packages and supporting libraries from `requirements.txt`.

| Package | Purpose |
|---------|---------|
| `langchain` | Core framework and LCEL pipeline |
| `langchain-community` | FAISS vector store integration |
| `langchain-huggingface` | Local HuggingFace embedding models |
| `langchain-google-genai` | Gemini LLM integration |
| `faiss-cpu` | Efficient vector similarity search |
| `sentence-transformers` | Downloads embedding model weights |
| `python-dotenv` | Loads API keys from `.env` |

In [1]:
! wget -O requirements.txt https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-langchain-00/requirements.txt
! pip install -r requirements.txt

--2026-06-12 04:23:07--  https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-langchain-00/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 121 [text/plain]
Saving to: ‘requirements.txt’

requirements.txt    100%[===================>]     121  --.-KB/s    in 0s      

2026-06-12 04:23:07 (3.30 MB/s) - ‘requirements.txt’ saved [121/121]



---
### Step 2: Define the Knowledge Base

These are the **source documents** the RAG system will search over — the "knowledge" it retrieves from.

In a real system these would be loaded from files, PDFs, or a database. Here we use a small list of plain strings about the fictional DQE department to keep the example focused on the RAG mechanics.

In [2]:
fictious_department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]

print(f"Knowledge base loaded: {len(fictious_department_info)} documents")

Knowledge base loaded: 8 documents


---
### Step 3: Load Environment Variables

Load API keys needed by the LLM and embedding services using `python-dotenv`.

- **`HF_TOKEN`** — HuggingFace API token (required for HuggingFace-hosted models)
- **`GEMINI_API_KEY`** — Google Gemini API key (required for the default LLM)

`load_dotenv()` reads these from a local `.env` file. In Google Colab, keys are read from **Secrets** as a fallback. The Gemini key is also mapped to `GOOGLE_API_KEY`, which is what `langchain-google-genai` expects internally.

In [3]:
import os
from dotenv import load_dotenv
from google.colab import userdata


# Load from .env file (local development)
#load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

# Fallback: try Google Colab secrets
#try:
   # if not os.getenv("HF_TOKEN"):
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
  #  if not os.getenv("GEMINI_API_KEY"):
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
#except ImportError:
    # Load from .env file (local development)
#    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = os.getenv("HF_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# LangChain Google GenAI requires GOOGLE_API_KEY
#if GEMINI_API_KEY:
#    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print(f"HF_TOKEN set:      {bool(HF_TOKEN)}")
print(f"GEMINI_API_KEY set: {bool(GEMINI_API_KEY)}")

HF_TOKEN set:      True
GEMINI_API_KEY set: True


---
### Step 4: Initialize the LLM (Model-Agnostic)

Use LangChain's `init_chat_model()` with the `"provider:model"` shorthand to initialize any supported LLM with a single line.

**Why model-agnostic?** Swapping the model string below is all that's needed to change providers — the retriever, prompt, and chain defined in later steps are completely unchanged.

| Provider | Model string |
|----------|-------------|
| **Google Gemini** (default) | `"google_genai:gemini-2.5-flash"` |
| HuggingFace TinyLlama | `"huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| OpenAI | `"openai:gpt-4o-mini"` |
| Groq | `"groq:llama-3.1-8b-instant"` |

In [6]:
from langchain.chat_models import init_chat_model
import os # Import os module to set environment variable

model = "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model_api_key=HF_TOKEN
#model = "google_genai:gemini-2.5-flash"
#model_api_key=GEMINI_API_KEY

# Ensure HUGGINGFACEHUB_API_TOKEN is set for HuggingFace models
os.environ["HUGGINGFACEHUB_API_TOKEN"] = model_api_key

llm = init_chat_model(
    model,
    api_key=model_api_key,
)

print(f"LLM initialized: {model}")

# To switch models, re-run this cell with a different model string:
#   init_chat_model("google_genai:gemini-2.5-flash-lite")          # Gemini (default)
#   init_chat_model("huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0")  # HuggingFace TinyLlama
#   init_chat_model("openai:gpt-4o-mini")                          # OpenAI
#   init_chat_model("groq:llama-3.1-8b-instant")                   # Groq


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM initialized: huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0


---
### Step 5: Create Embeddings and Vector Store

**Embeddings** are dense numerical vectors that capture the semantic meaning of text. Documents with similar meanings produce vectors that are close together in high-dimensional space — this is what enables semantic search.

1. **`HuggingFaceEmbeddings`** loads `all-MiniLM-L12-v2` locally — fast, lightweight, and no API key required
2. **`FAISS.from_texts()`** encodes every document and stores the resulting vectors in a FAISS index

> FAISS (Facebook AI Similarity Search) uses approximate nearest-neighbour search to find the most relevant documents in milliseconds, even across millions of vectors.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load open-source embedding model (runs locally, no API key needed)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2")
# Create FAISS vector store directly from text documents
vectorstore = FAISS.from_texts(fictious_department_info, embedding_model)

print(f"Loaded embedding model: all-MiniLM-L6-v2")
print(f"Number of documents indexed: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded embedding model: all-MiniLM-L6-v2
Number of documents indexed: 8


---
### Step 6: Create the Retriever

A **retriever** is the search interface over the vector store. Given a query string, it:

1. Embeds the query using the **same** embedding model used to index documents
2. Searches the FAISS index for the `k` closest vectors (cosine similarity)
3. Returns the matching text as `Document` objects

`k=2` means the **top 2 most semantically relevant** documents are returned for each query. Increasing `k` provides more context to the LLM but also increases prompt size and the risk of including less relevant content.

In [8]:
# Create retriever — returns top 2 most similar documents for any query
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

Retriever ready.


---
### Step 7: Define the Query

This is the natural-language question the RAG system will answer. Uncomment different queries to observe how retrieval and generation respond — especially notice how a question that falls outside the knowledge base is handled.

---
### Step 8: Build and Run the RAG Chain

Assemble the full pipeline using **LCEL** (LangChain Expression Language). The `|` pipe operator chains steps left-to-right, passing each output as the next step's input:

```
query
  ├─ retriever       →  fetch top-k relevant documents from FAISS
  ├─ format_docs     →  join document texts into one context string
  ├─ rag_prompt      →  inject {context} + {input} into the prompt template
  ├─ llm             →  generate a grounded answer from the LLM
  └─ StrOutputParser →  extract plain text from the LLM response object
```

The system prompt tells the LLM to **strictly use only the retrieved context** — this is what prevents hallucination and grounds answers in your knowledge base.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_system_prompt = (
    "Strictly use the provided documents to answer the user's question.\n\n"
    "Context:\n{context}"
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", rag_system_prompt),
    ("human", "{input}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL chain: retrieve → format → prompt → llm → parse
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Run RAG pipeline
user_query = "What is the Quantum AI Lab?"
retrieved_docs = retriever.invoke(user_query)
rag_answer = rag_chain.invoke(user_query)

print("Retrieved documents:")
for doc in retrieved_docs:
    print(f"  - {doc.page_content}")
print(f"\nRAG Answer: {rag_answer}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Retrieved documents:
  - The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.
  - DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.

RAG Answer: <|system|>
Strictly use the provided documents to answer the user's question.

Context:
The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.

DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.</s>
<|user|>
What is the Quantum AI Lab?</s>
<|assistant|>
The Quantum AI Lab is a research group at the National University of Singapore that investigates Quantum Neural Networks (QNNs) and collaborates with a national quantum computing cen

---
### Step 9: Compare — RAG vs. Direct LLM

Send the **same query** to the LLM without any retrieved context, then compare the two answers side by side.

| | With RAG | Without RAG |
|--|---------|------------|
| Context | Retrieved documents from the knowledge base | Only the LLM's training data |
| In-scope questions | Accurate, grounded answer | May match or hallucinate |
| Out-of-scope questions | Correctly says "I don't know" | Gives a generic or made-up answer |

This contrast is the core value proposition of RAG: **reliable, source-grounded responses**.

In [ ]:
# Compare: same question without RAG context
simple_response = llm.invoke([
    ("system", "You are a helpful assistant. You must always answer."),
    ("human", user_query),
])

print("=" * 60)
print(f"Query: {user_query}")
print("=" * 60)
print(f"\n[With RAG]    {rag_answer}")
print(f"\n[Without RAG] {simple_response.content}")